In [2]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from scipy.stats import zscore

from feature_engine.outliers import Winsorizer

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("/content/heart-disease.csv")

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [4]:
df.rename(columns={
    'chol':'cholesterol',
    'trestbps':'blood_pressure',
    'target':'disease_risk'
}, inplace=True)

df.head()

,age,sex,cp,blood_pressure,cholesterol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,disease_risk
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [5]:
np.random.seed(42)

regions = ['North','South','East','West']

df['patient_id'] = range(1, len(df)+1)

df['gender'] = df['sex'].map({1:'Male',0:'Female'})

df['region'] = np.random.choice(regions, len(df))

df['bmi'] = np.random.normal(26,5,len(df))

df['glucose'] = np.random.normal(110,20,len(df))

In [6]:
df = df[['patient_id',
         'age',
         'gender',
         'region',
         'bmi',
         'blood_pressure',
         'cholesterol',
         'glucose',
         'disease_risk']]

df.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,63,Male,East,27.690113,145,233,122.021079,1
1,2,37,Male,West,35.872856,130,250,97.114496,1
2,3,41,Female,North,35.825174,130,204,137.119408,1
3,4,56,Male,East,16.300444,120,236,95.207625,1
4,5,57,Female,East,22.186865,120,354,97.015054,1


In [9]:
mean_imputer = SimpleImputer(strategy='mean')

df['bmi_mean'] = mean_imputer.fit_transform(df[['bmi']])

In [25]:
mode_imputer = SimpleImputer(strategy='most_frequent')

df['region_mode'] = mode_imputer.fit_transform(df[['region']]).ravel()


In [11]:
df['gender_mode'] = mode_imputer.fit_transform(df[['gender']]).ravel()

In [23]:
df['bmi_missing'] = df['bmi'].isnull().astype(int)

random_samples = df['bmi'].dropna()

df.loc[df['bmi'].isnull(),'bmi'] = np.random.choice(
    random_samples,
    size=df['bmi'].isnull().sum()
)

In [13]:
knn_df = df[['age','bmi','blood_pressure','cholesterol','glucose']]

knn = KNNImputer(n_neighbors=5)

knn_result = pd.DataFrame(
    knn.fit_transform(knn_df),
    columns=knn_df.columns
)

knn_result.head()

,age,bmi,blood_pressure,cholesterol,glucose
0,63.0,18.275005,145.0,699.0,244.042158
1,37.0,27.144893,130.0,250.0,97.114496
2,41.0,35.825174,130.0,204.0,137.119408
3,56.0,16.300444,120.0,236.0,95.207625
4,57.0,22.186865,120.0,354.0,97.015054


In [14]:
mice = IterativeImputer(random_state=42)

mice_result = pd.DataFrame(
    mice.fit_transform(knn_df),
    columns=knn_df.columns
)

mice_result.head()

,age,bmi,blood_pressure,cholesterol,glucose
0,63.0,18.275005,145.0,699.0,244.042158
1,37.0,27.144893,130.0,250.0,97.114496
2,41.0,35.825174,130.0,204.0,137.119408
3,56.0,16.300444,120.0,236.0,95.207625
4,57.0,22.186865,120.0,354.0,97.015054


In [15]:
z_scores = np.abs(zscore(df[['cholesterol','glucose']].fillna(df.mean(numeric_only=True))))

outliers_z = (z_scores > 3).any(axis=1)

print("Outliers:", outliers_z.sum())

Outliers: 14


In [16]:
Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

iqr_outliers = df[(df['bmi']<lower) | (df['bmi']>upper)]

print("BMI Outliers:",len(iqr_outliers))

BMI Outliers: 2


In [17]:
lower_cap = df['cholesterol'].quantile(0.01)
upper_cap = df['cholesterol'].quantile(0.99)

df['cholesterol_percentile'] = np.clip(
    df['cholesterol'],
    lower_cap,
    upper_cap
)

In [19]:
winsor = Winsorizer(
    capping_method='quantiles',
    tail='both',
    fold=0.01,
    variables=['cholesterol','glucose'],
    missing_values='ignore'
)

df_winsor = winsor.fit_transform(df)

df_winsor.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk,bmi_mean,region_mode,gender_mode,bmi_missing,cholesterol_percentile
0,1,63,Male,East,18.275005,145,699.0,244.042158,1,25.944946,East,Male,1,699.0
1,2,37,Male,West,27.144893,130,250.0,97.114496,1,25.944946,West,Male,1,250.0
2,3,41,Female,North,35.825174,130,204.0,137.119408,1,35.825174,North,Female,0,204.0
3,4,56,Male,East,16.300444,120,236.0,95.207625,1,16.300444,East,Male,0,236.0
4,5,57,NaN,East,22.186865,120,354.0,97.015054,1,22.186865,East,Male,0,354.0


In [20]:
print("Original Shape:",df.shape)

print("Winsorized Shape:",df_winsor.shape)

Original Shape: (303, 14)
Winsorized Shape: (303, 14)


In [21]:
final_df = df_winsor.copy()

final_df.isnull().sum()

,0
patient_id,0
age,0
gender,30
region,30
bmi,0
blood_pressure,0
cholesterol,0
glucose,30
disease_risk,0
bmi_mean,0


In [27]:
# Replace original columns with imputed values

df['bmi'] = df['bmi_mean']

df['gender'] = df['gender_mode']

df['region'] = df['region_mode']

df['glucose'] = df['glucose'].fillna(df['glucose'].median())

In [28]:
final_df = df.copy()

print(final_df.isnull().sum())

patient_id                0
age                       0
gender                    0
region                    0
bmi                       0
blood_pressure            0
cholesterol               0
glucose                   0
disease_risk              0
bmi_mean                  0
region_mode               0
gender_mode               0
bmi_missing               0
cholesterol_percentile    0
dtype: int64
